In [2]:
import random
import operator
import math
import numpy as np
from deap import base, creator, tools

In [1]:
# Make a random individual.
def generate_individual(size, pos_min, pos_max):
    individual = creator.Individual(random.uniform(pos_min, pos_max) for _ in range(size))
    return individual


def mutate_adaptive_es(current_individual, sigma):
    # Add a random change to each variable.
    r = [random.gauss(0, sigma) for _ in range(len(current_individual))]

    # Make the new individual.
    new_individual = creator.Individual(
        list(
            map(
                operator.add,
                current_individual,
                r
            )
        )
    )

    return new_individual


# Fitness functions
def evaluation_rosenbrock(individual):
    fitness = sum(100 * (individual[i] ** 2 - individual[i + 1]) ** 2 + (individual[i] - 1) ** 2 for i in range(len(individual) - 1))
    return (fitness,)


def evaluation_griewank(individual):
    sum_part = sum(x ** 2 / 4000 for x in individual)
    product_part = math.prod(math.cos(x / math.sqrt(i + 1)) for i, x in enumerate(individual))

    fitness = sum_part - product_part + 1

    return (fitness,)

In [3]:
# Number of variables
NUM_DECISION_VARIABLES = 20

# Population size and number of generations
POPULATION_SIZE = 50
NUM_OF_GEN = 1000

# Search space limits
POS_MIN = -30
POS_MAX = 30

# Choose the fitness function here.
EVALUATION_FUNC_NAME = "evaluation_griewank"

EVALUATION_FUNCTIONS = {
    "evaluation_rosenbrock": evaluation_rosenbrock,
    "evaluation_griewank": evaluation_griewank
}

# Adaptive ES settings
ADAPTIVE_ES_C = 0.817
ADAPTIVE_ES_SIGMA = 10
ADAPTIVE_ES_WINDOW_LENGTH = min(NUM_DECISION_VARIABLES, 30)

In [4]:
# Set up the fitness and individual classes.
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

# Set up the toolbox.
toolbox = base.Toolbox()

toolbox.register("individual",
                 generate_individual,
                 size=NUM_DECISION_VARIABLES,
                 pos_min=POS_MIN,
                 pos_max=POS_MAX)

toolbox.register("mutate", mutate_adaptive_es)

# Use the selected fitness function.
toolbox.register("evaluate", EVALUATION_FUNCTIONS[EVALUATION_FUNC_NAME])

In [5]:
# Run the ES with different random seeds.
seed_list = list(range(1, 31))
results = []

for seed in seed_list:
    random.seed(seed)
    
    initial_individual = toolbox.individual()
    initial_individual.fitness.values = toolbox.evaluate(initial_individual)
    current_individual = initial_individual

    # Reset the ES values for each run.
    sigma = ADAPTIVE_ES_SIGMA
    mutation_success_history = []
    
    for gen in range(NUM_OF_GEN):
        is_mutation_success = 0
        # Make a new individual.
        new_mutated_individual = toolbox.mutate(current_individual, sigma)
        new_mutated_individual.fitness.values = toolbox.evaluate(new_mutated_individual)

        if (new_mutated_individual.fitness.values[0] < current_individual.fitness.values[0]):
            current_individual = new_mutated_individual
            is_mutation_success = 1
            
        mutation_success_history.append(is_mutation_success);
        
        if (len(mutation_success_history) >= ADAPTIVE_ES_WINDOW_LENGTH):
            recent_successes = mutation_success_history[-ADAPTIVE_ES_WINDOW_LENGTH:]
            count_success = sum(recent_successes)
            count_success_rate = count_success/ADAPTIVE_ES_WINDOW_LENGTH
            
            if count_success_rate > 0.2:
                # Increase sigma.
                sigma /= ADAPTIVE_ES_C ** 2

            elif count_success_rate < 0.2:
                # Decrease sigma.
                 sigma *= ADAPTIVE_ES_C ** 2
         
    results.append({
        "seed": seed,
        "best_solution": list(current_individual),
        "fitness": current_individual.fitness.values[0]
    })

    
    print(f'Seed : {seed}, function: {EVALUATION_FUNC_NAME}, Fitness: {current_individual.fitness.values[0]}')

Seed : 1, function: evaluation_griewank, Fitness: 0.44660147779221326
Seed : 2, function: evaluation_griewank, Fitness: 0.649785359657861
Seed : 3, function: evaluation_griewank, Fitness: 0.3103480039242361
Seed : 4, function: evaluation_griewank, Fitness: 1.0051855419690938
Seed : 5, function: evaluation_griewank, Fitness: 0.8973852266952611
Seed : 6, function: evaluation_griewank, Fitness: 0.6632919642970592
Seed : 7, function: evaluation_griewank, Fitness: 0.2995286227755175
Seed : 8, function: evaluation_griewank, Fitness: 0.5119512748667676
Seed : 9, function: evaluation_griewank, Fitness: 0.5238435095555263
Seed : 10, function: evaluation_griewank, Fitness: 0.49587984427311493
Seed : 11, function: evaluation_griewank, Fitness: 0.3739470659562707
Seed : 12, function: evaluation_griewank, Fitness: 0.7168504172864563
Seed : 13, function: evaluation_griewank, Fitness: 0.9237283815418347
Seed : 14, function: evaluation_griewank, Fitness: 0.5956878879584104
Seed : 15, function: evaluat

In [25]:
fitness_values = [result["fitness"] for result in results]

mean_fitness = np.mean(fitness_values)
std_fitness = np.std(fitness_values, ddof=1)

print("Mean:", mean_fitness)
print("Std:", std_fitness)

Mean: 0.6075269434507571
Std: 0.18980365738589824
